# 📈 Notebook 03 — Exploratory Data Analysis (EDA)

**Goal:** Uncover distribution patterns, trends, and regional differences through systematic visual exploration.

EDA is structured across the following checkpoints:

| Checkpoint | Focus |
|---|---|
| 1 | Global Sales Distribution (Univariate) |
| 2 | Sales by Genre (Univariate) |
| 3 | Sales by Platform — Top 15 (Univariate) |
| 4 | Annual Global Sales Trend (Time Series) |
| 5 | Regional Sales Breakdown (Regional) |
| 6 | Regional Genre Heatmap (Bivariate) |
| 7 | Sales Correlation Matrix (Multivariate) |
| 8 | Outlier Detection |

---

> **Pipeline context:** Raw data → Data Overview → Data Cleaning → Feature Engineering → **EDA** → Analysis

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Consistent visual style across all charts
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Libraries loaded.")

In [ ]:
from src.ingestion.load_dataset import load_raw_data
from src.cleaning.clean_sales_data import clean_sales_data
from src.features.feature_engineering import engineer_features

# Full pipeline: ingest → clean → feature engineering
df = engineer_features(clean_sales_data(load_raw_data()))
print(f"Dataset ready  →  {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

---

## 1 · Global Sales Distribution

Video game sales are highly right-skewed: a small number of blockbuster titles account for the majority of revenue.
The left chart shows the full distribution; the right zooms in on games selling fewer than 5 million copies (the vast majority).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full distribution (log-like tail)
sns.histplot(df["Global_Sales"], bins=60, ax=axes[0], color="steelblue")
axes[0].set_title("Global Sales Distribution (All Games)")
axes[0].set_xlabel("Global Sales (millions)")
axes[0].set_ylabel("Count")

# Zoomed view — games under 5M (covers the bulk of the market)
sns.histplot(df[df["Global_Sales"] < 5]["Global_Sales"], bins=60, ax=axes[1], color="steelblue")
axes[1].set_title("Global Sales Distribution (< 5M Units, Zoomed)")
axes[1].set_xlabel("Global Sales (millions)")
axes[1].set_ylabel("Count")

plt.suptitle("The Long Tail: Most Games Sell Under 1 Million Copies", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/01_global_sales_distribution.png", bbox_inches="tight")
plt.show()

---

## 2 · Sales by Genre

Which game genres generate the most cumulative sales?
Action and Sports dominate — partly because these genres have been staples across every console generation.

In [ ]:
genre_sales = df.groupby("Genre")["Global_Sales"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(genre_sales.index, genre_sales.values, color=sns.color_palette("muted", len(genre_sales)))
ax.set_title("Total Global Sales by Genre", fontsize=14, fontweight="bold")
ax.set_xlabel("Genre")
ax.set_ylabel("Total Sales (millions)")
plt.xticks(rotation=45, ha="right")

# Annotate top 3
for i, (x_pos, val) in enumerate(zip(range(3), genre_sales.values[:3])):
    ax.text(x_pos, val + 20, f"{val:,.0f}M", ha="center", fontsize=9, color="navy")

plt.tight_layout()
plt.savefig("../reports/figures/02_sales_by_genre.png", bbox_inches="tight")
plt.show()

---

## 3 · Sales by Platform — Top 15

Platform longevity and installed base size determine cumulative sales volumes.
The PlayStation 2 is the all-time leader — it had the largest library and a record-setting hardware install base.

In [ ]:
platform_sales = df.groupby("Platform")["Global_Sales"].sum().nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
platform_sales.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Platforms by Total Global Sales", fontsize=14, fontweight="bold")
ax.set_xlabel("Total Sales (millions)")
ax.set_ylabel("Platform")

# Annotate bars
for bar, val in zip(ax.patches, platform_sales.values):
    ax.text(val + 10, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}M", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("../reports/figures/03_sales_by_platform.png", bbox_inches="tight")
plt.show()

---

## 4 · Annual Global Sales Trend

How has the market changed over time?  
The rise of the gaming industry through the late 90s/early 2000s, peak around 2008–2009, and the decline in physical media sales are clearly visible.

In [ ]:
from src.analysis.market_analysis import sales_over_time, peak_sales_year

annual = sales_over_time(df)
peak_year = peak_sales_year(df)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(annual["Year"], annual["Global_Sales"], marker="o", linewidth=2, color="steelblue", markersize=4)
ax.axvline(x=peak_year, color="red", linestyle="--", linewidth=1.5, label=f"Peak: {peak_year}")
ax.fill_between(annual["Year"], annual["Global_Sales"], alpha=0.1, color="steelblue")
ax.set_title("Annual Global Video Game Sales (Physical, Retail)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Total Sales (millions)")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/04_annual_sales_trend.png", bbox_inches="tight")
plt.show()

---

## 5 · Regional Sales Breakdown

Not all regions engage with gaming equally. North America is the largest market, followed by Europe.
Japan — while a smaller absolute market — is notable for its unique genre preferences (covered next).

In [ ]:
from src.analysis.market_analysis import regional_totals

region = regional_totals(df)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart
axes[0].pie(region.values, labels=region.index, autopct="%1.1f%%", startangle=140,
            colors=sns.color_palette("pastel"))
axes[0].set_title("Regional Share of Global Sales")

# Bar chart with absolute values
axes[1].barh(region.index, region.values, color=sns.color_palette("muted"))
axes[1].set_xlabel("Total Sales (millions)")
axes[1].set_title("Total Sales by Region (millions)")
for bar, val in zip(axes[1].patches, region.values):
    axes[1].text(val + 10, bar.get_y() + bar.get_height() / 2,
                 f"{val:,.0f}M", va="center", fontsize=9)

plt.suptitle("North America Leads Global Video Game Sales", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/05_regional_share.png", bbox_inches="tight")
plt.show()

---

## 6 · Regional Genre Heatmap (Bivariate)

Which genres are most popular in each region?
This heatmap reveals **Japan's strong preference for Role-Playing games** compared to NA/EU, where Action and Sports dominate.

In [ ]:
from src.analysis.market_analysis import genre_sales_by_region

genre_region = genre_sales_by_region(df).set_index("Genre")
genre_region.columns = ["NA", "EU", "JP", "Other"]

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(genre_region, annot=True, fmt=".0f", cmap="YlOrRd", ax=ax,
            linewidths=0.5, linecolor="white")
ax.set_title("Genre Sales by Region (millions)", fontsize=14, fontweight="bold")
ax.set_xlabel("Region")
ax.set_ylabel("Genre")
plt.tight_layout()
plt.savefig("../reports/figures/06_genre_region_heatmap.png", bbox_inches="tight")
plt.show()

---

## 7 · Sales Correlation Matrix (Multivariate)

How strongly do different regional sales track with each other and with global totals?
High correlations (e.g. NA ↔ EU) suggest that successful games tend to be globally popular, not just regional hits.

In [ ]:
from src.analysis.market_analysis import genre_correlations

corr = genre_correlations(df)

fig, ax = plt.subplots(figsize=(7, 6))
mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
for i in range(len(corr)):
    for j in range(i):
        mask.iloc[i, j] = True  # show upper triangle only

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            ax=ax, mask=mask, linewidths=0.5, linecolor="white")
ax.set_title("Regional & Global Sales Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/figures/07_sales_correlation.png", bbox_inches="tight")
plt.show()

---

## 8 · Outlier Detection

Statistical outliers (> 3σ above the mean) represent blockbuster games that far outperform the rest of the market.
These are important to identify because they can distort averages and bias platform/genre comparisons.

In [ ]:
from src.analysis.market_analysis import detect_outliers

outliers = detect_outliers(df)
print(f"Outlier games (> 3σ above mean global sales): {len(outliers)}\n")

top_outliers = outliers[["Name", "Platform", "Publisher", "Genre", "Global_Sales"]].sort_values(
    "Global_Sales", ascending=False
)
display(top_outliers)

# Box plot to visualise spread
fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(df["Global_Sales"], vert=False, patch_artist=True,
           boxprops=dict(facecolor="steelblue", color="navy"),
           flierprops=dict(marker="o", color="red", markersize=4, alpha=0.5))
ax.set_title("Global Sales Distribution — Box Plot (Outliers in Red)", fontsize=13, fontweight="bold")
ax.set_xlabel("Global Sales (millions)")
ax.set_yticks([])
plt.tight_layout()
plt.savefig("../reports/figures/08_outlier_boxplot.png", bbox_inches="tight")
plt.show()

---

## 📝 EDA Summary

| Finding | Detail |
|---|---|
| Sales distribution | Highly right-skewed — most games sell under 1M units |
| Top genre | Action leads globally, followed by Sports and Shooter |
| Top platform | PS2 dominates cumulative sales across all eras |
| Market peak | ~2008–2009 for physical retail sales |
| Largest region | North America (~49% of global sales) |
| Japan preference | Role-Playing games dominate JP vs. Action/Sports in NA & EU |
| Correlation | NA ↔ EU sales are strongly correlated; JP shows weaker correlation with western markets |
| Blockbusters | A small number of outlier titles (Wii Sports, GTA V, etc.) vastly outperform the market |

**Next:** [`04_analysis.ipynb`](04_analysis.ipynb) — answer the core analytical questions and construct the final narrative.